In [8]:
!pip install pandas
!pip install -U pandas transformers accelerate huggingface_hub sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 304.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 587.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 575.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 643.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 641.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 118.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.8/807.8 kB 905.2 MB/s  0:00:00
  Attempting uninstall: idna0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/18 [regex]
    Found existing installation: idna 3.10━━━━━━━━━━━━━━━━━━━━  4/18 [regex]
    Uninstalling idna-3.10:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/18 [regex]
      Successfully uninstalled idna-3.10━━━━━━━━━━━━━━━━━━━━━━  4/18 [regex]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [transformers] [transformers]ub]


In [9]:
# ==========================================
# Step 1. LLM 평가 환경 확인
# ==========================================

import torch
import pandas as pd
import transformers

print("===== LLM Evaluation Environment =====")
print("PyTorch version :", torch.__version__)
print("Transformers version :", transformers.__version__)
print("Pandas version :", pd.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print(
        "GPU VRAM :",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

===== LLM Evaluation Environment =====
PyTorch version : 2.8.0+cu128
Transformers version : 5.18.0
Pandas version : 3.0.6
CUDA available : True
GPU : NVIDIA A100 80GB PCIe
GPU VRAM : 79.25 GB


In [19]:
# ==========================================
# Step 2. LLM 평가 데이터셋 생성
# ==========================================

evaluation_data = [
    {
        "id": 1,
        "category": "설명",
        "prompt": "생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘."
    },
    {
        "id": 2,
        "category": "요약",
        "prompt": """다음 내용을 한 문장으로 요약해줘.
GPU는 대규모 병렬 연산에 특화된 프로세서이다.
딥러닝에서는 행렬 연산이 많이 사용되기 때문에
GPU를 활용하면 모델 학습과 추론 속도를 높일 수 있다."""
    },
    {
        "id": 3,
        "category": "지시준수",
        "prompt": "오픈소스 LLM의 장점을 정확히 3개만 번호를 붙여 작성해줘."
    },
    {
        "id": 4,
        "category": "구조화 출력",
        "prompt": """Qwen과 Gemma를 비교하는 다음 정보를 JSON 형식으로만 출력해줘.
필드: model, developer, usage
설명이나 Markdown은 추가하지 마."""
    },
    {
        "id": 5,
        "category": "분류",
        "prompt": """다음 문장을 긍정, 부정, 중립 중 하나로만 분류해줘.
'GPU를 사용하니 모델 실행 속도가 크게 향상되었다.'"""
    }
]

evaluation_df = pd.DataFrame(evaluation_data)

evaluation_df

,id,category,prompt
0,1,설명,생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘.
1,2,요약,다음 내용을 한 문장으로 요약해줘.\nGPU는 대규모 병렬 연산에 특화된 프로세서이...
2,3,지시준수,오픈소스 LLM의 장점을 정확히 3개만 번호를 붙여 작성해줘.
3,4,구조화 출력,Qwen과 Gemma를 비교하는 다음 정보를 JSON 형식으로만 출력해줘.\n필드:...
4,5,분류,"다음 문장을 긍정, 부정, 중립 중 하나로만 분류해줘.\n'GPU를 사용하니 모델 ..."


In [10]:
print("model 변수 존재 :", "model" in globals())
print("tokenizer 변수 존재 :", "tokenizer" in globals())

model 변수 존재 : False
tokenizer 변수 존재 : False


Gemma 다시 로드

In [11]:
# ==========================================
# Step 3-1. Gemma 3 4B 평가 모델 로드
# ==========================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-3-4b-it"
cache_dir = "/root/hf-cache"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    cache_dir=cache_dir
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=cache_dir
)

model.eval()

print("Model :", model_id)
print("GPU :", torch.cuda.get_device_name(0))
print(
    "VRAM Allocated :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model : google/gemma-3-4b-it
GPU : NVIDIA A100 80GB PCIe
VRAM Allocated : 8.01 GB


자동 평가 함수 작성

In [17]:
# ==========================================
# Step 3-2. LLM 자동 평가 함수
# ==========================================

import time
import torch

def evaluate_model(prompt, model, tokenizer):

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt}
            ]
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    input_tokens = inputs["input_ids"].shape[1]
    generated_tokens = outputs.shape[1] - input_tokens

    response = tokenizer.decode(
        outputs[0][input_tokens:],
        skip_special_tokens=True
    ).strip()

    tokens_per_sec = generated_tokens / elapsed

    peak_vram = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    return {
        "response": response,
        "generated_tokens": generated_tokens,
        "latency_sec": round(elapsed, 3),
        "tokens_per_sec": round(tokens_per_sec, 2),
        "peak_vram_gb": round(peak_vram, 2)
    }

print("평가 함수 준비 완료")

평가 함수 준비 완료


TEST

In [20]:
test_result = evaluate_model(
    evaluation_data[0]["prompt"],
    model,
    tokenizer
)

test_result

{'response': "생성형 AI는 기존 인공지능과 달리 데이터를 학습하여 새로운 콘텐츠를 창작해내는 반면, 기존 인공지능은 주어진 데이터를 분석하고 패턴을 인식하여 예측이나 분류와 같은 특정 작업을 수행합니다. 즉, 생성형 AI는 '만들기'에 집중하는 반면, 기존 인공지능은 '알기'에 집중한다고 볼 수 있습니다. 이러한 차이 때문에 생성형 AI는 텍스트, 이미지, 오디오 등 다양한 형태의 콘텐츠 제작에 활용될 수 있는 반면, 기존 인공지능은 주로 데이터 분석, 자동화, 의사 결정 등 특정 분야에서 활용됩니다.",
 'generated_tokens': 137,
 'latency_sec': 7.162,
 'tokens_per_sec': 19.13,
 'peak_vram_gb': 8.04}

Gemma 5개 문항 자동 평가

In [21]:
# ==========================================
# Step 4. Gemma 전체 평가 실행
# ==========================================

gemma_results = []

for item in evaluation_data:

    print(
        f'평가 중: {item["id"]} - {item["category"]}'
    )

    result = evaluate_model(
        item["prompt"],
        model,
        tokenizer
    )

    gemma_results.append({
        "id": item["id"],
        "category": item["category"],
        "prompt": item["prompt"],
        "response": result["response"],
        "generated_tokens": result["generated_tokens"],
        "latency_sec": result["latency_sec"],
        "tokens_per_sec": result["tokens_per_sec"],
        "peak_vram_gb": result["peak_vram_gb"]
    })

print("Gemma 전체 평가 완료")

평가 중: 1 - 설명
평가 중: 2 - 요약
평가 중: 3 - 지시준수
평가 중: 4 - 구조화 출력
평가 중: 5 - 분류
Gemma 전체 평가 완료


결과를 DataFrame으로 확인

In [22]:
gemma_df = pd.DataFrame(gemma_results)

gemma_df[
    [
        "id",
        "category",
        "generated_tokens",
        "latency_sec",
        "tokens_per_sec",
        "peak_vram_gb"
    ]
]

,id,category,generated_tokens,latency_sec,tokens_per_sec,peak_vram_gb
0,1,설명,137,6.332,21.64,8.04
1,2,요약,43,2.055,20.93,8.03
2,3,지시준수,197,9.723,20.26,8.05
3,4,구조화 출력,95,4.627,20.53,8.04
4,5,분류,3,0.155,19.37,8.03


Valid Output(출력 품질) 자동 평가

In [23]:
# ==========================================
# Step 5-1. 모델 응답 확인
# ==========================================

for _, row in gemma_df.iterrows():
    print("=" * 60)
    print(f'[{row["id"]}] {row["category"]}')
    print(row["response"])
    print()

[1] 설명
생성형 AI는 기존 인공지능과 달리 데이터를 학습하여 새로운 콘텐츠를 창작해내는 반면, 기존 인공지능은 주어진 데이터를 분석하고 패턴을 인식하여 예측이나 분류와 같은 특정 작업을 수행합니다. 즉, 생성형 AI는 '만들기'에 집중하는 반면, 기존 인공지능은 '알기'에 집중한다고 볼 수 있습니다. 이러한 차이 때문에 생성형 AI는 텍스트, 이미지, 오디오 등 다양한 형태의 콘텐츠 제작에 활용될 수 있는 반면, 기존 인공지능은 주로 데이터 분석, 자동화, 의사 결정 등 특정 분야에서 활용됩니다.

[2] 요약
GPU는 대규모 병렬 연산에 특화되어 딥러닝에서 행렬 연산의 속도를 향상시켜 모델 학습 및 추론을 가속화하는 데 효과적입니다.

[3] 지시준수
오픈소스 LLM의 주요 장점 3가지입니다.

1.  **투명성과 검증 가능성:** 오픈소스 모델은 소스 코드와 학습 데이터가 공개되어 있어 내부 작동 방식을 이해하고 검증할 수 있습니다. 이는 모델의 편향성, 보안 취약점 등을 파악하고 개선하는 데 도움이 됩니다.
2.  **맞춤화 및 유연성:** 사용자는 자신의 특정 요구사항에 맞춰 모델을 수정하고 재학습시킬 수 있습니다. 이는 특정 도메인이나 작업에 최적화된 모델을 만들거나, 기존 모델의 성능을 향상시키는 데 유용합니다.
3.  **비용 효율성:** 상용 LLM API를 사용하는 것보다 오픈소스 모델을 자체적으로 운영하면 장기적으로 비용을 절감할 수 있습니다. 또한, 커뮤니티 기반의 협업을 통해 개발 비용을 분담할 수 있습니다.

[4] 구조화 출력
```json
[
  {
    "model": "Qwen",
    "developer": "Alibaba",
    "usage": "Large language model, available through Alibaba Cloud and other platforms."
  },
  {
    "model": "Gemma",
    "developer": "Google DeepMind",
    "usag

In [24]:
# ==========================================
# Step 5-2. Structured Output 자동 검증
# ==========================================

import json

json_response = gemma_df.loc[
    gemma_df["category"] == "구조화 출력",
    "response"
].iloc[0]

try:
    parsed = json.loads(json_response)
    json_valid = True
except json.JSONDecodeError:
    parsed = None
    json_valid = False

print("===== JSON 출력 검증 =====")
print("Valid JSON :", json_valid)

if json_valid:
    print("Parsed JSON :", parsed)

===== JSON 출력 검증 =====
Valid JSON : False


In [25]:
# ==========================================
# Step 5-3. Markdown 제거 후 JSON 재검증
# ==========================================

import json

clean_json = json_response.strip()

if clean_json.startswith("```json"):
    clean_json = clean_json[7:]

if clean_json.endswith("```"):
    clean_json = clean_json[:-3]

clean_json = clean_json.strip()

try:
    parsed = json.loads(clean_json)
    cleaned_json_valid = True
except json.JSONDecodeError:
    parsed = None
    cleaned_json_valid = False

print("원본 출력 Valid JSON :", False)
print("Markdown 제거 후 Valid JSON :", cleaned_json_valid)

if cleaned_json_valid:
    print("\nParsed JSON:")
    print(parsed)

원본 출력 Valid JSON : False
Markdown 제거 후 Valid JSON : True

Parsed JSON:
[{'model': 'Qwen', 'developer': 'Alibaba', 'usage': 'Large language model, available through Alibaba Cloud and other platforms.'}, {'model': 'Gemma', 'developer': 'Google DeepMind', 'usage': 'Open-weights language model, available for research and commercial use.'}]


평가 결과표 만들기

In [26]:
# ==========================================
# Step 6. 출력 품질 평가
# ==========================================

import json
import re
import pandas as pd

quality_results = []

for _, row in gemma_df.iterrows():

    category = row["category"]
    response = row["response"].strip()

    instruction_pass = None
    valid_output = None
    note = ""

    # 1. 설명: 정확히 3문장인지 확인
    if category == "설명":
        sentences = [
            s.strip()
            for s in re.split(r'[.!?]+', response)
            if s.strip()
        ]

        instruction_pass = len(sentences) == 3
        note = f"문장 수: {len(sentences)}"

    # 2. 요약: 한 문장인지 확인
    elif category == "요약":
        sentences = [
            s.strip()
            for s in re.split(r'[.!?]+', response)
            if s.strip()
        ]

        instruction_pass = len(sentences) == 1
        note = f"문장 수: {len(sentences)}"

    # 3. 지시준수: 번호가 붙은 3개 항목인지 확인
    elif category == "지시준수":

        numbered_items = re.findall(
            r'(?m)^\s*\d+[\.\)]\s+',
            response
        )

        instruction_pass = len(numbered_items) == 3
        note = f"번호 항목 수: {len(numbered_items)}"

    # 4. 구조화 출력: 원본 자체가 Valid JSON인지 확인
    elif category == "구조화 출력":

        try:
            json.loads(response)
            valid_output = True
        except json.JSONDecodeError:
            valid_output = False

        instruction_pass = valid_output

        if valid_output:
            note = "Raw JSON 파싱 성공"
        else:
            note = "Markdown 등 추가 출력으로 Raw JSON 파싱 실패"

    # 5. 분류: 지정된 세 단어 중 하나만 출력했는지 확인
    elif category == "분류":

        instruction_pass = response in [
            "긍정",
            "부정",
            "중립"
        ]

        valid_output = instruction_pass
        note = f"출력: {response}"

    quality_results.append({
        "id": row["id"],
        "category": category,
        "instruction_pass": instruction_pass,
        "valid_output": valid_output,
        "note": note
    })

quality_df = pd.DataFrame(quality_results)

quality_df

,id,category,instruction_pass,valid_output,note
0,1,설명,True,None,문장 수: 3
1,2,요약,True,None,문장 수: 1
2,3,지시준수,True,None,번호 항목 수: 3
3,4,구조화 출력,False,False,Markdown 등 추가 출력으로 Raw JSON 파싱 실패
4,5,분류,True,True,출력: 긍정


성능 데이터와 합친다

In [27]:
final_gemma_df = gemma_df.merge(
    quality_df,
    on=["id", "category"]
)

final_gemma_df[
    [
        "id",
        "category",
        "latency_sec",
        "tokens_per_sec",
        "peak_vram_gb",
        "instruction_pass",
        "valid_output",
        "note"
    ]
]

,id,category,latency_sec,tokens_per_sec,peak_vram_gb,instruction_pass,valid_output,note
0,1,설명,6.332,21.64,8.04,True,None,문장 수: 3
1,2,요약,2.055,20.93,8.03,True,None,문장 수: 1
2,3,지시준수,9.723,20.26,8.05,True,None,번호 항목 수: 3
3,4,구조화 출력,4.627,20.53,8.04,False,False,Markdown 등 추가 출력으로 Raw JSON 파싱 실패
4,5,분류,0.155,19.37,8.03,True,True,출력: 긍정


Qwen3-8B를 동일한 평가 데이터와 동일한 평가 함수로 테스트하기 위해
Gemma를 GPU에서 제거

In [28]:
# ==========================================
# Step 7-1. Gemma GPU 메모리 해제
# ==========================================

import gc
import torch

del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU Memory after cleanup :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

GPU Memory after cleanup : 0.01 GB


Qwen3-8B 로드

In [29]:
# ==========================================
# Step 7-2. Qwen3-8B BF16 모델 로드
# ==========================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen3-8B"
cache_dir = "/root/hf-cache"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    cache_dir=cache_dir
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=cache_dir
)

model.eval()

print("Model :", model_id)
print("GPU :", torch.cuda.get_device_name(0))
print(
    "VRAM Allocated :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model : Qwen/Qwen3-8B
GPU : NVIDIA A100 80GB PCIe
VRAM Allocated : 15.26 GB


Qwen3-8B를 Gemma와 동일 조건으로 평가
Thinking Mode를 끄기

In [30]:
# ==========================================
# Step 7-3. Qwen3 자동 평가 함수
# ==========================================

import time
import torch

def evaluate_qwen(prompt, model, tokenizer):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Qwen3 Thinking Mode OFF
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    input_tokens = inputs["input_ids"].shape[1]
    generated_tokens = outputs.shape[1] - input_tokens

    response = tokenizer.decode(
        outputs[0][input_tokens:],
        skip_special_tokens=True
    ).strip()

    tokens_per_sec = generated_tokens / elapsed

    peak_vram = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    return {
        "response": response,
        "generated_tokens": generated_tokens,
        "latency_sec": round(elapsed, 3),
        "tokens_per_sec": round(tokens_per_sec, 2),
        "peak_vram_gb": round(peak_vram, 2)
    }

print("Qwen3 평가 함수 준비 완료")

Qwen3 평가 함수 준비 완료


In [ ]:
TEST

In [31]:
test_qwen = evaluate_qwen(
    evaluation_data[0]["prompt"],
    model,
    tokenizer
)

test_qwen

{'response': '생성형 AI는 입력 데이터를 기반으로 새로운 내용을 창조하는 능력을 가진 인공지능입니다. 기존 인공지능은 주로 데이터를 분석하거나 특정 작업을 수행하는 데 초점을 맞추고 있습니다. 생성형 AI는 창의성과 유연성을 통해 새로운 아이디어나 콘텐츠를 만들어내는 것이 특징입니다.',
 'generated_tokens': 90,
 'latency_sec': 3.622,
 'tokens_per_sec': 24.85,
 'peak_vram_gb': 15.28}

In [ ]:
5개 평가문항 전체 실행

In [32]:
# ==========================================
# Step 7-5. Qwen3-8B 전체 평가
# ==========================================

qwen_results = []

for item in evaluation_data:

    print(
        f'평가 중: {item["id"]} - {item["category"]}'
    )

    result = evaluate_qwen(
        item["prompt"],
        model,
        tokenizer
    )

    qwen_results.append({
        "id": item["id"],
        "category": item["category"],
        "prompt": item["prompt"],
        "response": result["response"],
        "generated_tokens": result["generated_tokens"],
        "latency_sec": result["latency_sec"],
        "tokens_per_sec": result["tokens_per_sec"],
        "peak_vram_gb": result["peak_vram_gb"]
    })

print("Qwen3-8B 전체 평가 완료")

평가 중: 1 - 설명
평가 중: 2 - 요약
평가 중: 3 - 지시준수
평가 중: 4 - 구조화 출력
평가 중: 5 - 분류
Qwen3-8B 전체 평가 완료


표로 확인하기

In [33]:
qwen_df = pd.DataFrame(qwen_results)

qwen_df[
    [
        "id",
        "category",
        "generated_tokens",
        "latency_sec",
        "tokens_per_sec",
        "peak_vram_gb"
    ]
]

,id,category,generated_tokens,latency_sec,tokens_per_sec,peak_vram_gb
0,1,설명,90,3.325,27.07,15.28
1,2,요약,55,2.098,26.21,15.28
2,3,지시준수,200,7.633,26.20,15.30
3,4,구조화 출력,84,3.244,25.89,15.28
4,5,분류,3,0.124,24.12,15.28


Qwen 출력 품질을 Gemma와 동일하게 평가

In [34]:
# ==========================================
# Step 8. Qwen3 출력 품질 평가
# ==========================================

import json
import re
import pandas as pd

qwen_quality_results = []

for _, row in qwen_df.iterrows():

    category = row["category"]
    response = row["response"].strip()

    instruction_pass = None
    valid_output = None
    note = ""

    # 1. 설명: 정확히 3문장
    if category == "설명":

        sentences = [
            s.strip()
            for s in re.split(r'[.!?]+', response)
            if s.strip()
        ]

        instruction_pass = len(sentences) == 3
        note = f"문장 수: {len(sentences)}"

    # 2. 요약: 정확히 1문장
    elif category == "요약":

        sentences = [
            s.strip()
            for s in re.split(r'[.!?]+', response)
            if s.strip()
        ]

        instruction_pass = len(sentences) == 1
        note = f"문장 수: {len(sentences)}"

    # 3. 지시준수: 번호가 붙은 3개 항목
    elif category == "지시준수":

        numbered_items = re.findall(
            r'(?m)^\s*\d+[\.\)]\s+',
            response
        )

        instruction_pass = len(numbered_items) == 3
        note = f"번호 항목 수: {len(numbered_items)}"

    # 4. 구조화 출력: Raw JSON 검증
    elif category == "구조화 출력":

        try:
            json.loads(response)
            valid_output = True
        except json.JSONDecodeError:
            valid_output = False

        instruction_pass = valid_output

        if valid_output:
            note = "Raw JSON 파싱 성공"
        else:
            note = "Raw JSON 파싱 실패"

    # 5. 분류: 지정된 단어 하나만 출력
    elif category == "분류":

        instruction_pass = response in [
            "긍정",
            "부정",
            "중립"
        ]

        valid_output = instruction_pass
        note = f"출력: {response}"

    qwen_quality_results.append({
        "id": row["id"],
        "category": category,
        "instruction_pass": instruction_pass,
        "valid_output": valid_output,
        "note": note
    })

qwen_quality_df = pd.DataFrame(qwen_quality_results)

final_qwen_df = qwen_df.merge(
    qwen_quality_df,
    on=["id", "category"]
)

final_qwen_df[
    [
        "id",
        "category",
        "latency_sec",
        "tokens_per_sec",
        "peak_vram_gb",
        "instruction_pass",
        "valid_output",
        "note"
    ]
]

,id,category,latency_sec,tokens_per_sec,peak_vram_gb,instruction_pass,valid_output,note
0,1,설명,3.325,27.07,15.28,True,None,문장 수: 3
1,2,요약,2.098,26.21,15.28,True,None,문장 수: 1
2,3,지시준수,7.633,26.20,15.30,True,None,번호 항목 수: 3
3,4,구조화 출력,3.244,25.89,15.28,False,False,Raw JSON 파싱 실패
4,5,분류,0.124,24.12,15.28,True,True,출력: 긍정


Qwen vs Gemma 최종 비교표

In [35]:
# ==========================================
# Step 9. Qwen3-8B vs Gemma 3 4B 최종 비교
# ==========================================

# 모델 이름 추가
qwen_compare = final_qwen_df.copy()
qwen_compare["model"] = "Qwen3-8B"

gemma_compare = final_gemma_df.copy()
gemma_compare["model"] = "Gemma-3-4B-IT"

# 필요한 열만 선택
compare_df = pd.concat([
    qwen_compare[
        [
            "model",
            "id",
            "category",
            "latency_sec",
            "tokens_per_sec",
            "peak_vram_gb",
            "instruction_pass",
            "valid_output"
        ]
    ],
    gemma_compare[
        [
            "model",
            "id",
            "category",
            "latency_sec",
            "tokens_per_sec",
            "peak_vram_gb",
            "instruction_pass",
            "valid_output"
        ]
    ]
], ignore_index=True)

compare_df

,model,id,category,latency_sec,tokens_per_sec,peak_vram_gb,instruction_pass,valid_output
0,Qwen3-8B,1,설명,3.325,27.07,15.28,True,None
1,Qwen3-8B,2,요약,2.098,26.21,15.28,True,None
2,Qwen3-8B,3,지시준수,7.633,26.20,15.30,True,None
3,Qwen3-8B,4,구조화 출력,3.244,25.89,15.28,False,False
4,Qwen3-8B,5,분류,0.124,24.12,15.28,True,True
5,Gemma-3-4B-IT,1,설명,6.332,21.64,8.04,True,None
6,Gemma-3-4B-IT,2,요약,2.055,20.93,8.03,True,None
7,Gemma-3-4B-IT,3,지시준수,9.723,20.26,8.05,True,None
8,Gemma-3-4B-IT,4,구조화 출력,4.627,20.53,8.04,False,False
9,Gemma-3-4B-IT,5,분류,0.155,19.37,8.03,True,True


평균 성능도 계산

In [36]:
# ==========================================
# Step 9-2. 모델별 평균 성능
# ==========================================

summary_df = compare_df.groupby("model").agg(
    avg_latency_sec=("latency_sec", "mean"),
    avg_tokens_per_sec=("tokens_per_sec", "mean"),
    avg_peak_vram_gb=("peak_vram_gb", "mean")
).round(2)

summary_df

,avg_latency_sec,avg_tokens_per_sec,avg_peak_vram_gb
model,,,
Gemma-3-4B-IT,4.58,20.55,8.04
Qwen3-8B,3.28,25.90,15.28


지시준수율 계산

In [37]:
instruction_summary = compare_df.groupby("model").agg(
    instruction_tests=("instruction_pass", "count"),
    instruction_passes=("instruction_pass", "sum")
)

instruction_summary["instruction_pass_rate"] = (
    instruction_summary["instruction_passes"]
    / instruction_summary["instruction_tests"]
    * 100
).round(1)

instruction_summary

,instruction_tests,instruction_passes,instruction_pass_rate
model,,,
Gemma-3-4B-IT,5,4,80.0
Qwen3-8B,5,4,80.0


프로젝트 모델 선정 기준 만들기

In [38]:
# ==========================================
# Step 10-1. 모델 선정용 핵심 지표 생성
# ==========================================

selection_summary = compare_df.groupby("model").agg(
    avg_speed=("tokens_per_sec", "mean"),
    avg_peak_vram=("peak_vram_gb", "mean"),
    instruction_pass_rate=("instruction_pass", "mean")
).reset_index()

selection_summary["instruction_pass_rate"] *= 100

selection_summary = selection_summary.round(2)

selection_summary

,model,avg_speed,avg_peak_vram,instruction_pass_rate
0,Gemma-3-4B-IT,20.55,8.04,80.0
1,Qwen3-8B,25.90,15.28,80.0


Structured Output 결과도 추가

In [39]:
# ==========================================
# Step 10-2. Structured Output 평가 추가
# ==========================================

json_results = compare_df[
    compare_df["category"] == "구조화 출력"
][
    ["model", "valid_output"]
].copy()

json_results = json_results.rename(
    columns={"valid_output": "raw_json_valid"}
)

selection_summary = selection_summary.merge(
    json_results,
    on="model",
    how="left"
)

selection_summary

,model,avg_speed,avg_peak_vram,instruction_pass_rate,raw_json_valid
0,Gemma-3-4B-IT,20.55,8.04,80.0,False
1,Qwen3-8B,25.90,15.28,80.0,False


프로젝트 요구사항별 해석표 만들기

In [40]:
# ==========================================
# Step 10-3. 프로젝트 요구사항별 모델 특성
# ==========================================

project_selection = pd.DataFrame({
    "프로젝트 요구사항": [
        "빠른 응답이 중요한 서비스",
        "GPU 메모리가 제한된 환경",
        "일반적인 지시 준수",
        "Raw JSON 기반 자동화"
    ],
    "판단 기준": [
        "tokens/sec",
        "Peak VRAM",
        "Instruction Pass Rate",
        "Valid JSON"
    ],
    "실험에서 확인할 내용": [
        "평균 생성 처리량 비교",
        "평균 Peak VRAM 비교",
        "5개 교육용 테스트의 지시 준수 결과 비교",
        "후처리 없이 JSON 파싱 가능한지 확인"
    ]
})

project_selection

,프로젝트 요구사항,판단 기준,실험에서 확인할 내용
0,빠른 응답이 중요한 서비스,tokens/sec,평균 생성 처리량 비교
1,GPU 메모리가 제한된 환경,Peak VRAM,평균 Peak VRAM 비교
2,일반적인 지시 준수,Instruction Pass Rate,5개 교육용 테스트의 지시 준수 결과 비교
3,Raw JSON 기반 자동화,Valid JSON,후처리 없이 JSON 파싱 가능한지 확인


평가 결과를 CSV로 저장하여 results/ 폴더에 남기고, GitHub에 올릴 수 있는 프로젝트 산출물 형태로 완성

결과 저장 폴더 만들기

In [41]:
# ==========================================
# Step 11-1. 평가 결과 저장 폴더 생성
# ==========================================

import os

results_dir = "/workspace/results"
os.makedirs(results_dir, exist_ok=True)

print("Results directory :", results_dir)

Results directory : /workspace/results


Qwen·Gemma 전체 평가 결과 CSV 저장

In [42]:
# ==========================================
# Step 11-2. 전체 평가 결과 저장
# ==========================================

compare_path = os.path.join(
    results_dir,
    "model_evaluation_results.csv"
)

compare_df.to_csv(
    compare_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료 :", compare_path)

저장 완료 : /workspace/results/model_evaluation_results.csv


모델별 요약 결과 저장

In [43]:
# ==========================================
# Step 11-3. 모델 선정 요약 결과 저장
# ==========================================

summary_path = os.path.join(
    results_dir,
    "model_selection_summary.csv"
)

selection_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료 :", summary_path)

저장 완료 : /workspace/results/model_selection_summary.csv


실제 저장 여부 확인

In [44]:
# ==========================================
# Step 11-4. 저장 파일 확인
# ==========================================

for filename in os.listdir(results_dir):
    print(filename)

model_evaluation_results.csv
model_selection_summary.csv


In [45]:
check_df = pd.read_csv(
    "/workspace/results/model_evaluation_results.csv"
)

check_df.head()

,model,id,category,latency_sec,tokens_per_sec,peak_vram_gb,instruction_pass,valid_output
0,Qwen3-8B,1,설명,3.325,27.07,15.28,True,NaN
1,Qwen3-8B,2,요약,2.098,26.21,15.28,True,NaN
2,Qwen3-8B,3,지시준수,7.633,26.20,15.30,True,NaN
3,Qwen3-8B,4,구조화 출력,3.244,25.89,15.28,False,False
4,Qwen3-8B,5,분류,0.124,24.12,15.28,True,True


Notebook 저장